# LAB: Chaos Agent: Tools, Lore, Questionable Judgment

## Objective

Build a **creative complaint handler** for *NormalObjects* with a LangChain agent.
The agent picks its own tools, so you must **prove** the answer comes from tool output.

```
complaint → agent (LLM) → picks tools (any order) → tool outputs → final answer
```

## Steps

| # | Step | Code |
|---|------|---|
| 1 | Setup | PROVIDED |
| 2 | Create creative tools | PROVIDED + YOUR TURN |
| 3 | Create the agent | YOUR TURN |
| 4 | Test with sample complaints | YOUR TURN |
| 5 | Analyze agent behavior | YOUR TURN |
| | `lab_proof.md` | YOUR TURN |

**Submit:** `lab_proof.md` with code, one complaint, tool evidence, final output, one failure.

| Label | Meaning |
|---|---|
| **PROVIDED** | Run and read it |
| **YOUR TURN** | Replace the `# TODO` lines |
| **OPTIONAL** | Not required |

**Reference:** `langchain_integration.ipynb` (tools + `create_agent`)

**Prompt discipline:** use AI to review or debug, not to write your answer.

---
## Step 1: Setup [PROVIDED]

Add `OPENAI_API_KEY=...` to a `.env` file next to this notebook.

In [1]:
# Install required packages (uncomment to run)
# !pip install -U langchain langchain-openai python-dotenv

In [2]:
import os
import random
from dotenv import load_dotenv

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

load_dotenv()

# Make sure OPENAI_API_KEY is set in your .env file
print("OpenAI API Key loaded:", "OPENAI_API_KEY" in os.environ)

model = init_chat_model(model="gpt-4o-mini", model_provider="openai", temperature=0.7)

OpenAI API Key loaded: True


**Done when:** the key prints `True` and there are no import errors.

---
## Step 2: Create Creative Tools

Every tool needs:
1. the `@tool` decorator
2. a clear docstring (the agent reads it to choose the tool)

### 2a. `consult_demogorgon` [PROVIDED]

In [3]:
@tool
def consult_demogorgon(complaint: str) -> str:
    """Consult the Demogorgon about a complaint involving the Upside Down.
    Returns a chaotic, unreliable opinion."""
    responses = [
        f"The Demogorgon sniffs '{complaint}' and roars. It seems... hungry, not helpful.",
        f"The Demogorgon tilts its petal-face at '{complaint}'. Interpreted as: 'turn off the lights'.",
        f"The Demogorgon ignores '{complaint}' and walks back into the wall. Classic.",
    ]
    return random.choice(responses)

### 2b. `check_hawkins_records` [YOUR TURN]

- Dictionary with 4 keys: `portal`, `monsters`, `psychics`, `electricity`
- Return the record whose key appears in the query (case-insensitive)
- No match → `"No record found ..."`

In [4]:
@tool
def check_hawkins_records(query: str) -> str:
    """Look up official Hawkins town records.
    Covers only these topics: portal, monsters, psychics, electricity.
    Put one of these words in the query. Returns the record or 'No record found'."""
    records = {
        "portal": "Record H-01: A gate to the Upside Down opened under Hawkins Lab in 1983. Sealed, but reopened twice.",
        "monsters": "Record H-02: Confirmed creatures: the Demogorgon and Demodogs. They hunt by smell and avoid fire.",
        "psychics": "Record H-03: Subject 011 shows telekinesis. Nosebleeds follow heavy use.",
        "electricity": "Record H-04: Flickering lights mark activity from the Upside Down nearby.",
    }
    for key, record in records.items():
        if key in query.lower():
            return record
    return f"No record found for '{query}'. Records exist only for: portal, monsters, psychics, electricity."

### 2c. `cast_interdimensional_spell` [YOUR TURN]

- Args: `problem`, `creativity_level="medium"`
- Map `low/medium/high` → `1/2/3`
- 4 spell strings that include the problem
- `random.sample()` N spells, join with `"\n"`

In [5]:
@tool
def cast_interdimensional_spell(problem: str, creativity_level: str = "medium") -> str:
    """Cast creative spells to solve a problem.
    creativity_level: 'low' (1 spell), 'medium' (2 spells) or 'high' (3 spells)."""
    levels = {"low": 1, "medium": 2, "high": 3}
    n = levels.get(creativity_level.lower(), 2)
    spells = [
        f"🕯 Circle of Christmas lights: surround '{problem}' with blinking bulbs until it gets bored.",
        f"🎸 Play loud 80s music at '{problem}'. Upside Down creatures hate a good guitar solo.",
        f"🔥 Fire ward: draw a ring of flames around '{problem}' (adult supervision required).",
        f"📻 Tune a radio to static and whisper '{problem}' backwards three times.",
    ]
    return "\n".join(random.sample(spells, n))

### 2d. `gather_party_wisdom` [YOUR TURN]

- Dictionary with 4 keys, one per party member (Mike, Dustin, Lucas, Will)
- Return the match (case-insensitive)
- No match → generic "the party huddles..." answer

In [6]:
@tool
def gather_party_wisdom(question: str) -> str:
    """Ask the party (Mike, Dustin, Lucas, Will) for advice and a plan.
    Name a member in the question to hear from them; otherwise the whole party answers."""
    party = {
        "mike": "Mike: 'We stick together. Nobody goes into the basement alone.'",
        "dustin": "Dustin: 'Science first. Find the source of the signal, then we act.'",
        "lucas": "Lucas: 'Scout it, keep the slingshot ready, and have an exit route.'",
        "will": "Will: 'Trust the cold feeling. If the back of your neck tingles, it's close.'",
    }
    for name, advice in party.items():
        if name in question.lower():
            return advice
    return "The party huddles: 'Gather everyone, grab flashlights and walkie-talkies, and check the records first.'"

In [7]:
print(consult_demogorgon.invoke({"complaint": "My lights keep flickering"}))
print(check_hawkins_records.invoke({"query": "tell me about the portal"}))
print(check_hawkins_records.invoke({"query": "my cat is weird"}))      # fallback test
print(cast_interdimensional_spell.invoke({"problem": "Demodogs in my garden", "creativity_level": "high"}))
print(gather_party_wisdom.invoke({"question": "what is the plan?"}))

The Demogorgon ignores 'My lights keep flickering' and walks back into the wall. Classic.
Record H-01: A gate to the Upside Down opened under Hawkins Lab in 1983. Sealed, but reopened twice.
No record found for 'my cat is weird'. Records exist only for: portal, monsters, psychics, electricity.
🕯 Circle of Christmas lights: surround 'Demodogs in my garden' with blinking bulbs until it gets bored.
🔥 Fire ward: draw a ring of flames around 'Demodogs in my garden' (adult supervision required).
📻 Tune a radio to static and whisper 'Demodogs in my garden' backwards three times.
The party huddles: 'Gather everyone, grab flashlights and walkie-talkies, and check the records first.'


**Done when:** all 4 tools return text, and a query with no matching key returns your fallback.

---
## Step 3: Create the Agent [YOUR TURN]

System prompt must:
- encourage creative tool use
- force at least one tool call
- say: answer **only** from tool results
- ask the agent to list the tools it used

**Hint:** `create_agent(model=..., tools=..., system_prompt=...)` in `langchain_integration.ipynb`.

In [8]:
tools = [consult_demogorgon, check_hawkins_records, cast_interdimensional_spell, gather_party_wisdom]

system_prompt = """You are the NormalObjects complaint handler for strange events in Hawkins.

Rules:
1. Always call at least one tool before answering. Never answer from your own knowledge.
2. Be creative: combine several tools if it helps (records for facts, party for a plan, spells for creative fixes, the Demogorgon for chaos).
3. Use ONLY information that appears in tool results. If a tool says "No record found", say so clearly. Do not invent a record.
4. The Demogorgon is unreliable: present its opinion as chaos, not as fact.
5. End your answer with a line: "Tools used: ..." listing every tool you called."""

agent = create_agent(model=model, tools=tools, system_prompt=system_prompt)

print([t.name for t in tools])

['consult_demogorgon', 'check_hawkins_records', 'cast_interdimensional_spell', 'gather_party_wisdom']


**Done when:** the agent is created and prints the 4 tool names.

---
## Step 4: Test with Sample Complaints [YOUR TURN]

Use `config={"recursion_limit": 10}` to stop loops.

In [9]:
complaints = [
    "My Christmas lights keep flickering and I hear noises from the wall. What is going on?",
    "There is a weird portal in my basement. What's the plan?",
    "A Demodog ate my homework. I need a creative solution, creativity level high!",
    "My neighbor's cat is acting weird. Is it from the Upside Down?",  # no record exists
]

results = []
for complaint in complaints:
    result = agent.invoke(
        {"messages": [HumanMessage(content=complaint)]},
        config={"recursion_limit": 10},
    )
    results.append(result)
    print("🗣 COMPLAINT:", complaint)
    print("🤖 ANSWER:", result["messages"][-1].content)
    print("-" * 80)

🗣 COMPLAINT: My Christmas lights keep flickering and I hear noises from the wall. What is going on?
🤖 ANSWER: It appears that your flickering Christmas lights might be linked to activity from the Upside Down nearby. According to Hawkins town records, there was a gate to the Upside Down that opened under Hawkins Lab in 1983, which has been sealed but reopened twice since then. 

In a more chaotic twist, the Demogorgon seems to react to your situation with a roar, indicating it might be hungry, but unfortunately, it doesn't provide any helpful insights.

You may want to consider checking for any strange disturbances or anomalies in your home that could be causing this flickering and noise, as it could indicate a breach or influx from the Upside Down.

Tools used: check_hawkins_records, consult_demogorgon
--------------------------------------------------------------------------------
🗣 COMPLAINT: There is a weird portal in my basement. What's the plan?
🤖 ANSWER: According to the Hawkins 

In [10]:
print(results[3]["messages"][-1].content)

There are no records in Hawkins that specifically mention anything about cats, so I can't confirm if your neighbor's cat is linked to the Upside Down. However, the Demogorgon suggests that the cat's strange behavior could be related to turning off the lights. This chaotic opinion might imply that the cat is reacting to something unusual in its environment.

If you suspect something more sinister, it might be worth investigating any unusual happenings around your neighbor's house. 

Tools used: check_hawkins_records, consult_demogorgon.


In [11]:
import textwrap
print(textwrap.fill(results[3]["messages"][-1].content, 120))

There are no records in Hawkins that specifically mention anything about cats, so I can't confirm if your neighbor's cat
is linked to the Upside Down. However, the Demogorgon suggests that the cat's strange behavior could be related to
turning off the lights. This chaotic opinion might imply that the cat is reacting to something unusual in its
environment.  If you suspect something more sinister, it might be worth investigating any unusual happenings around your
neighbor's house.   Tools used: check_hawkins_records, consult_demogorgon.


**Check:** for the cat complaint, does the agent admit "no record", or invent one?

---
## Step 5: Analyze Agent Behavior [YOUR TURN]

### 5a. Track tool usage

For each result: print the tool order (`a → b → c`). Then count total uses per tool.

**Hint:** AI messages have `.tool_calls`; use `collections.Counter`.

In [12]:
from collections import Counter

counts = Counter()
for i, r in enumerate(results, 1):
    order = []
    for msg in r["messages"]:
        if msg.type == "ai":
            for call in msg.tool_calls:
                order.append(call["name"])
    counts.update(order)
    print(f"{i}. " + " → ".join(order))

print("\nTotal uses per tool:", dict(counts))

1. check_hawkins_records → check_hawkins_records → consult_demogorgon
2. check_hawkins_records → gather_party_wisdom
3. consult_demogorgon → cast_interdimensional_spell
4. check_hawkins_records → consult_demogorgon

Total uses per tool: {'check_hawkins_records': 4, 'consult_demogorgon': 3, 'gather_party_wisdom': 1, 'cast_interdimensional_spell': 1}


### 5b. Trace one answer (grounding)

For one result, print every tool call, its args, and what it returned.

**Hint:** messages with `msg.type == "tool"` hold the tool output.

In [13]:
r = results[2]   # the homework complaint: most interesting one

for msg in r["messages"]:
    if msg.type == "ai" and msg.tool_calls:
        for call in msg.tool_calls:
            print(f"🔧 CALL: {call['name']}  ARGS: {call['args']}")
    elif msg.type == "tool":
        print(f"📤 RESULT from {msg.name}:\n{msg.content}\n")

print("💬 FINAL ANSWER:\n", textwrap.fill(r["messages"][-1].content, 120))

🔧 CALL: consult_demogorgon  ARGS: {'complaint': 'A Demodog ate my homework.'}
🔧 CALL: cast_interdimensional_spell  ARGS: {'problem': 'A Demodog ate my homework.', 'creativity_level': 'high'}
📤 RESULT from consult_demogorgon:
The Demogorgon tilts its petal-face at 'A Demodog ate my homework.'. Interpreted as: 'turn off the lights'.

📤 RESULT from cast_interdimensional_spell:
🎸 Play loud 80s music at 'A Demodog ate my homework.'. Upside Down creatures hate a good guitar solo.
📻 Tune a radio to static and whisper 'A Demodog ate my homework.' backwards three times.
🕯 Circle of Christmas lights: surround 'A Demodog ate my homework.' with blinking bulbs until it gets bored.

💬 FINAL ANSWER:
 To tackle the issue of your homework being devoured by a Demodog, I've consulted the Demogorgon, which whimsically
suggests you should "turn off the lights."   For a more creative fix, I recommend the following high-level
interdimensional spell:  1. **Play loud 80s music**: Upside Down creatures, includi

**Check:** mark each claim in the final answer as either from a tool or invented.

### 5c. Compare with a structured approach

Write `structured_handler(complaint)`: always calls records → party → spell (`low`), no LLM.

In [16]:
def structured_handler(complaint: str) -> str:
    record = check_hawkins_records.invoke({"query": complaint})
    wisdom = gather_party_wisdom.invoke({"question": complaint})
    spell = cast_interdimensional_spell.invoke({"problem": complaint, "creativity_level": "low"})
    return f"📁 Records: {record}\n👥 Party: {wisdom}\n✨ Spell: {spell}"

print(structured_handler(complaints[1]))
print()
print(structured_handler(complaints[0]))

📁 Records: Record H-01: A gate to the Upside Down opened under Hawkins Lab in 1983. Sealed, but reopened twice.
👥 Party: The party huddles: 'Gather everyone, grab flashlights and walkie-talkies, and check the records first.'
✨ Spell: 🎸 Play loud 80s music at 'There is a weird portal in my basement. What's the plan?'. Upside Down creatures hate a good guitar solo.

📁 Records: No record found for 'My Christmas lights keep flickering and I hear noises from the wall. What is going on?'. Records exist only for: portal, monsters, psychics, electricity.
👥 Party: The party huddles: 'Gather everyone, grab flashlights and walkie-talkies, and check the records first.'
✨ Spell: 📻 Tune a radio to static and whisper 'My Christmas lights keep flickering and I hear noises from the wall. What is going on?' backwards three times.


| | Agent | Structured |
|---|---|---|
| Same tools every run? | No | Yes |
| Tool order predictable? | No | Yes |
| Can invent facts? | Yes (adds explanations and promises) | No (only tool text) |
| Easy to trace? | Needs the message trace | Yes, the output i sthe trace |
| Best for which client case? | Open, varied requests, with human review | Regulated, repeatable processes that must be auditable |

---
## Extensions [OPTIONAL]

- More tools (`consult_eleven`, `check_government_files`)
- LLM call inside a tool
- Memory with `InMemorySaver` + `thread_id`
- Streamlit / Gradio UI
- Measure response time and tool calls per complaint

In [ ]:
# YOUR TURN (optional)

# TODO

---
## Write `lab_proof.md` [YOUR TURN]

```markdown
# Lab Proof: NormalObjects Creative Complaint Handler

## 1. What I built
## 2. Query
## 3. Tool evidence (calls + args + outputs)
## 4. Final output
## 5. Trace (which claim comes from which tool output?)
## 6. Failure or limitation (where can it give a plausible but unsupported answer?)
## 7. Agent vs. structured handler
```

**Before you submit:**
- [ ] No API keys or `.env`
- [ ] Only lab files
- [ ] `lab_proof.md` is the entry point

---
### Troubleshooting

| Problem | Fix |
|---|---|
| Agent doesn't use tools | Clearer docstrings; prompt must require tool use |
| Tools in "wrong" order | Expected for agents. Need fixed order? Use LangGraph |
| Agent loops | Lower `recursion_limit`, improve docstrings |
| Generic answers | More specific tool outputs, lower temperature |
| `ImportError` on `create_agent` | `pip install -U langchain` (needs v1+) |